In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Path to dataset files:", path)

Path to dataset files: /home/nikita/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


In [2]:
import os

file = os.listdir(path)
files_in_dir = os.listdir(path + '/' + file[0])

print(files_in_dir)

path_to_files = path + '/' + file[0]

['genres_original', 'features_3_sec.csv', 'features_30_sec.csv', 'images_original']


In [3]:
path_to_csv_3 =  path_to_files + '/' + files_in_dir[1]
path_to_csv_3

'/home/nikita/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data/features_3_sec.csv'

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(path_to_csv_3)

df['time'] = df['filename'].str[-5].astype('int')
df = df.drop(['filename','length'], axis=1)

lab = {l: i for i,l in enumerate(np.unique(df['label']))}

df['res'] = df['label'].map(lab).astype('int')

df = df.drop(['label'], axis=1)

df

,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,rolloff_mean,rolloff_var,...,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,time,res
0,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,3714.560359,1.080790e+06,...,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,0,0
1,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,3869.682242,6.722448e+05,...,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,1,0
2,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,3997.639160,7.907127e+05,...,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,2,0
3,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,3568.300218,9.216524e+05,...,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,3,0
4,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,3469.992864,6.102111e+05,...,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9985,0.349126,0.080515,0.050019,0.000097,1499.083005,164266.886443,1718.707215,85931.574523,3015.559458,8.479527e+05,...,-9.094270,38.326839,-4.246976,31.049839,-5.625813,48.804092,1.818823,38.966969,5,9
9986,0.372564,0.082626,0.057897,0.000088,1847.965128,281054.935973,1906.468492,99727.037054,3746.694524,1.170890e+06,...,-12.375726,66.418587,-3.081278,54.414265,-11.960546,63.452255,0.428857,18.697033,6,9
9987,0.347481,0.089019,0.052403,0.000701,1346.157659,662956.246325,1561.859087,138762.841945,2442.362154,2.602871e+06,...,-2.524483,21.778994,4.809936,25.980829,1.775686,48.582378,-0.299545,41.586990,7,9
9988,0.387527,0.084815,0.066430,0.000320,2084.515327,203891.039161,2018.366254,22860.992562,4313.266226,4.968878e+05,...,-5.363541,17.209942,6.462601,21.442928,2.354765,24.843613,0.675824,12.787750,8,9


In [5]:
from sklearn.preprocessing import MinMaxScaler

X = df.drop(['res'], axis=1).to_numpy()
y = df['res'].to_numpy()

scaler = MinMaxScaler()
scaler.fit(X)
X_scal = scaler.transform(X)

In [6]:
X_new = []
y_new = []

temp = []
for i in range(len(X_scal)-1):
    temp.append(X_scal[i])
    if round(X_scal[:,-1][i+1],1) == 0 and round(X_scal[:,-1][i],1) == 1:
        X_new.append(temp)
        y_new.append(y[i])
        temp = []
    elif round(X_scal[:,-1][i+1],1) == 0 and round(X_scal[:,-1][i],1) == 0.9:
        temp = []

X_new = np.array(X_new).astype('float32')
y_new = np.array(y_new).astype('float32')


In [7]:
x_train = []
x_test = []
y_train = []
y_test = []

for i in range(len(y_new)):
    if i%9 == 0:
        x_test.append(X_new[i])
        y_test.append(y_new[i])
    else:
        x_train.append(X_new[i])
        y_train.append(y_new[i])

x_train = np.array(x_train).astype('float32')
x_test = np.array(x_test).astype('float32')
y_train = np.array(y_train).astype('float32')
y_test = np.array(y_test).astype('float32')

In [8]:
x_train.shape

(879, 10, 58)

In [9]:
x_train[0::2].shape

(440, 10, 58)

In [10]:
import collections 
from collections import Counter

Counter(y_train), Counter(y_test)

(Counter({5.0: 89,
          6.0: 89,
          7.0: 89,
          0.0: 88,
          1.0: 88,
          3.0: 88,
          8.0: 88,
          4.0: 87,
          9.0: 87,
          2.0: 86}),
 Counter({0.0: 12,
          8.0: 12,
          2.0: 11,
          3.0: 11,
          4.0: 11,
          5.0: 11,
          6.0: 11,
          7.0: 11,
          1.0: 10,
          9.0: 10}))

In [12]:
import tensorflow as tf
import keras
from tensorflow.keras.callbacks import EarlyStopping

print("Версия TensorFlow:", tf.__version__)
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("Устройство для вычислений:", tf.test.is_gpu_available())

model = keras.Sequential([
    keras.layers.GRU(32, return_sequences=False, activation='tanh', input_shape=(x_train.shape[1], x_train.shape[2])),
    keras.layers.Dense(10, activation ='softmax')
])

# model.summary()


model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 5e-4),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping = EarlyStopping(
                    monitor='val_loss',
                    patience = 50,
                    min_delta = 0.001,
                    verbose = 0,
                    restore_best_weights = True,
                    mode = 'min'
                    )

model.fit(
        x = x_train,
        y = y_train,
        batch_size = 16,
        epochs = 2000,
        verbose = 0,
        callbacks = [early_stopping], 
        shuffle = True,
        # validation_split = 0.05,
        # validation_data = None,
        validation_split = 0,
        validation_data = (x_test[0::2], y_test[0::2]),
        validation_batch_size = 8
        )

train_loss, keras_train_r2 = model.evaluate(x_train, y_train)
test_loss, keras_test_r2 = model.evaluate(x_test, y_test)
print('\nдля RNN точность для обучающей \ тестовой выборки:', round(keras_train_r2, 2), ' \ ', round(keras_test_r2, 2))

Версия TensorFlow: 2.14.0
Доступные GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Устройство для вычислений: True


2026-05-01 12:36:21.123646: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-01 12:36:21.124652: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-01 12:36:21.125428: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

4/4 [==============================] - 0s 2ms/step - loss: 0.8373 - accuracy: 0.7091

для RNN точность для обучающей \ тестовой выборки: 0.84  \  0.71


In [13]:
base_path = path_to_files + '/' + files_in_dir[3]

files_list = os.listdir(base_path)

data_files = []

for i in range(len(files_list)):
    class_path = os.path.join(base_path, files_list[i])
    path_obj = os.listdir(class_path)
    path_obj_full = [os.path.join(class_path, f) for f in path_obj]
    data_files.append(path_obj_full)

In [14]:
from PIL import Image
import random

data_x = []
data_y = []
size = (260, 260)

for class_idx, file_list in enumerate(data_files):
    num_samples = len(file_list)
    
    for j in range(num_samples):
        img = Image.open(file_list[j])
        res = img.convert('RGB').resize(size)
        res = np.array(res) / 255.0
        data_x.append(res)
        data_y.append(class_idx)

In [15]:
from sklearn.model_selection import train_test_split

data_x = np.array(data_x)
data_y = np.array(data_y)

x_train, x_test, y_train, y_test = train_test_split(
    data_x, 
    data_y, 
    test_size=0.10, 
    random_state=42,
    stratify=data_y
)

In [16]:
del data_x
del data_y

In [17]:
import keras
from tensorflow.keras.callbacks import EarlyStopping
from keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras import layers

In [24]:
model = keras.Sequential([
    keras.Input(shape = (260, 260, 3)),
    keras.layers.Conv2D(5, (3, 3), padding = 'same', strides = 2),
    keras.layers.MaxPooling2D((2, 4)),
    keras.layers.Conv2D(5, (3, 3), padding = 'same', activation ='relu', strides = 2),
    keras.layers.MaxPooling2D((2, 4)),
    keras.layers.Flatten(),
    keras.layers.Dense(32),
    keras.layers.Dense(64, activation ='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(32, activation ='relu'),
    keras.layers.Dense(10, activation ='softmax')
])

model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-4),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    patience = 50,
                    min_delta = 0.01,
                    verbose = 0,
                    restore_best_weights = True
                    )

In [25]:
model.fit(
        x = x_train,
        y = y_train,
        batch_size = 16,
        epochs = 2000,
        verbose = 0,
        callbacks = [early_stopping], 
        shuffle = True,
        validation_split = 0.1,
        validation_data = None,
        validation_batch_size = 8
        )

In [26]:
train_loss, keras_train_acc = model.evaluate(x_train, y_train)
test_loss, keras_test_acc = model.evaluate(x_test, y_test)
print('\nдля CNN точность для обучающей \ тестовой выборки:', round(keras_train_acc, 2), ' \ ', round(keras_test_acc, 2))

4/4 [==============================] - 0s 4ms/step - loss: 1.4297 - accuracy: 0.5300

для CNN точность для обучающей \ тестовой выборки: 0.68  \  0.53
